In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
from datasets import load_from_disk

data_dir = "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-b"
ds = load_from_disk(data_dir)

print(ds)
print("Train columns:", ds["train"].column_names)
print("Validation columns:", ds["validation"].column_names)
print("Sample:", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 500000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 100000
    })
})
Train columns: ['label', 'input_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'attention_mask']
Sample: {'label': 0, 'input_ids': [0, 729, 2613, 126, 1250, 130, 10522, 130, 1751, 953, 317, 377, 462, 1679, 132, 7673, 144, 317, 394, 7343, 132, 393, 2925, 3792, 196, 1367, 9733, 869, 3801, 595, 18996, 1478, 126, 3239, 132, 1478, 181, 2473, 126, 9731, 1543, 17782, 1587, 8535, 784, 317, 377, 462, 800, 1679, 132, 1918, 144, 317, 394, 1568, 144, 317, 411, 6817, 385, 1751, 132, 814, 429, 317, 394, 3552, 144, 317, 411, 7343, 132, 393, 2925, 503, 13014, 6634, 434, 35331, 3636, 3041, 17782, 1587, 882, 534, 9346, 147, 1736, 127, 317, 411, 6817, 385, 1938, 317, 377, 1568, 144, 317, 394, 3192, 181, 636, 385, 3192, 132, 1030, 126, 2012, 

In [3]:
!pip install -U transformers datasets evaluate accelerate pyarrow scikit-learn

In [4]:
import os

data_dir = "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-b"

print("Danh sách file:")
for f in os.listdir(data_dir):
    print("-", f)

Danh sách file:
- dataset_dict.json
- train
- validation


In [5]:
new_train_code = r'''
import os
import json
import argparse
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import evaluate


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model_name", type=str, default="microsoft/unixcoder-base")
    parser.add_argument("--max_train_samples", type=int, default=None)
    parser.add_argument("--max_eval_samples", type=int, default=None)
    parser.add_argument("--num_train_epochs", type=int, default=3)
    parser.add_argument("--per_device_train_batch_size", type=int, default=4)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=4)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=4)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--logging_steps", type=int, default=3000)
    parser.add_argument("--save_total_limit", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--bf16", action="store_true")
    parser.add_argument("--metric_for_best_model", type=str, default="f1")
    return parser.parse_args()


def detect_label_column(dataset):
    candidates = ["labels", "label", "target", "class", "y"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Không tìm thấy cột nhãn. Các cột hiện có: {dataset.column_names}")


def detect_input_columns(dataset):
    has_ids = "input_ids" in dataset.column_names
    has_mask = "attention_mask" in dataset.column_names
    has_token_type_ids = "token_type_ids" in dataset.column_names

    if not has_ids:
        raise ValueError(f"Không tìm thấy cột input_ids. Các cột hiện có: {dataset.column_names}")

    return has_mask, has_token_type_ids


def infer_label_info(train_ds, label_col):
    values = train_ds[label_col]
    unique_labels = sorted(list(set(values)))

    if all(isinstance(x, (int, np.integer)) for x in unique_labels):
        label2id = {str(int(v)): int(v) for v in unique_labels}
        id2label = {int(v): str(int(v)) for v in unique_labels}
        num_labels = len(unique_labels)
    else:
        unique_labels = sorted([str(v) for v in unique_labels])
        label2id = {v: i for i, v in enumerate(unique_labels)}
        id2label = {i: v for v, i in label2id.items()}
        num_labels = len(unique_labels)

    return num_labels, label2id, id2label


def encode_labels(example, label_col, label2id):
    value = example[label_col]
    if isinstance(value, (int, np.integer)) and str(int(value)) in label2id:
        example["labels"] = int(label2id[str(int(value))])
    elif str(value) in label2id:
        example["labels"] = int(label2id[str(value)])
    else:
        raise ValueError(f"Nhãn không hợp lệ: {value}")
    return example


class TokenizedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        import torch

        labels = [f["labels"] for f in features]
        features_no_labels = [{k: v for k, v in f.items() if k != "labels"} for f in features]

        batch = self.tokenizer.pad(
            features_no_labels,
            padding=True,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


def build_compute_metrics():
    accuracy_metric = evaluate.load("accuracy")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        accuracy = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
        precision = precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
        recall = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]
        f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    return compute_metrics


def main():
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)

    print("Đang load dataset từ:", args.data_dir)
    dataset_dict = load_from_disk(args.data_dir)

    label_col = detect_label_column(dataset_dict["train"])
    num_labels, label2id, id2label = infer_label_info(dataset_dict["train"], label_col)

    processed = {}
    for split in dataset_dict.keys():
        ds = dataset_dict[split].map(
            lambda x: encode_labels(x, label_col, label2id),
            desc=f"Encoding labels for {split}",
        )

        keep_cols = ["input_ids", "attention_mask", "labels"]
        remove_cols = [c for c in ds.column_names if c not in keep_cols]
        ds = ds.remove_columns(remove_cols)
        processed[split] = ds

    train_dataset = processed["train"]
    eval_dataset = processed["validation"]

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        args.model_name,
        num_labels=num_labels,
        label2id=label2id,
        id2label=id2label,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name,
        config=config,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.num_train_epochs,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        logging_steps=args.logging_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=args.save_total_limit,
        save_steps=31250,
        load_best_model_at_end=True,
        metric_for_best_model=args.metric_for_best_model,
        greater_is_better=True,
        fp16=args.fp16,
        bf16=args.bf16,
        seed=args.seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=TokenizedDataCollator(tokenizer),
        compute_metrics=build_compute_metrics(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    best_model_dir = os.path.join(args.output_dir, "best_model")
    os.makedirs(best_model_dir, exist_ok=True)

    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)

    with open(os.path.join(best_model_dir, "label2id.json"), "w", encoding="utf-8") as f:
        json.dump(label2id, f, ensure_ascii=False, indent=2)

    with open(os.path.join(best_model_dir, "id2label.json"), "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

    eval_metrics = trainer.evaluate(eval_dataset=eval_dataset)
    print("Validation metrics:", eval_metrics)

    meta = {
        "model_name": args.model_name,
        "task_name": os.path.basename(args.data_dir),
        "num_labels": num_labels,
        "label_column": label_col,
        "train_size": len(train_dataset),
        "validation_size": len(eval_dataset),
        "best_metric_name": args.metric_for_best_model,
        "validation_metrics": eval_metrics,
    }

    with open(os.path.join(best_model_dir, "model_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("Da luu model tai:", best_model_dir)


if __name__ == "__main__":
    main()
'''
with open("train_unixcoder.py", "w", encoding="utf-8") as f:
    f.write(new_train_code)

print("Đã cập nhật train_unixcoder.py")

Đã cập nhật train_unixcoder.py


In [6]:
!python train_unixcoder.py \
  --data_dir "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-b" \
  --output_dir "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_b" \
  --num_train_epochs 3 \
  --per_device_train_batch_size 4 \
  --per_device_eval_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-5 \
  --weight_decay 0.01 \
  --logging_steps 3000 \
  --fp16

Streaming output truncated to the last 5000 lines.
 76% 19013/25000 [05:19<01:37, 61.48it/s]
 76% 19020/25000 [05:19<01:37, 61.60it/s]
 76% 19027/25000 [05:19<01:37, 61.14it/s]
 76% 19034/25000 [05:19<01:38, 60.70it/s]
 76% 19041/25000 [05:20<01:38, 60.32it/s]
 76% 19048/25000 [05:20<01:38, 60.38it/s]
 76% 19055/25000 [05:20<01:38, 60.33it/s]
 76% 19062/25000 [05:20<01:38, 60.56it/s]
 76% 19069/25000 [05:20<01:38, 60.39it/s]
 76% 19076/25000 [05:20<01:37, 60.79it/s]
 76% 19083/25000 [05:20<01:38, 59.91it/s]
 76% 19089/25000 [05:20<01:39, 59.22it/s]
 76% 19095/25000 [05:21<01:39, 59.20it/s]
 76% 19101/25000 [05:21<01:39, 59.26it/s]
 76% 19108/25000 [05:21<01:39, 59.43it/s]
 76% 19115/25000 [05:21<01:38, 59.49it/s]
 76% 19122/25000 [05:21<01:38, 59.92it/s]
 77% 19129/25000 [05:21<01:37, 60.04it/s]
 77% 19136/25000 [05:21<01:37, 60.21it/s]
 77% 19143/25000 [05:21<01:37, 60.34it/s]
 77% 19150/25000 [05:21<01:36, 60.53it/s]
 77% 19157/25000 [05:22<01:36, 60.59it/s]
 77% 19164/25000 [05:22<0

In [7]:
output_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_b/best_model"

print("Các file trong best_model:")
for f in os.listdir(output_dir):
    print("-", f)

Các file trong best_model:
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json
- training_args.bin
- label2id.json
- id2label.json
- model_meta.json


In [8]:
from transformers import AutoTokenizer

model_name = "microsoft/unixcoder-base"
best_model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_b/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(best_model_dir)

print("Đã lưu lại tokenizer đầy đủ")
print(os.listdir(best_model_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Đã lưu lại tokenizer đầy đủ
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'label2id.json', 'id2label.json', 'model_meta.json']


In [9]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_b/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with open(f"{model_dir}/id2label.json", "r", encoding="utf-8") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

code_text = """
def add(a, b):
    return a + b
"""

inputs = tokenizer(
    code_text,
    truncation=True,
    padding=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

print({
    "predicted_label_id": pred_id,
    "predicted_label": id2label[pred_id],
    "confidence": float(probs[pred_id])
})

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'predicted_label_id': 7, 'predicted_label': '7', 'confidence': 0.6708219647407532}


In [10]:

  #--resume_checkpoint "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/checkpoint-36000" \